# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

Import Library

In [2]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


Get Dataset and Change Path

In [6]:
import kagglehub
import shutil
import os

# Download dataset
path = kagglehub.dataset_download(
    "albertnathaniel12/indonesian-spices-dataset"
)

print("Path dataset asli:", path)

# Direktori tujuan di MacBook
destination = os.path.expanduser(
    "/Users/akbarrzql/PycharmProjects/IndonesiaSpicesProject/data"
)

# Buat parent directory jika belum tersedia
os.makedirs(os.path.dirname(destination), exist_ok=True)

# Hapus folder lama jika sudah ada
if os.path.exists(destination):
    shutil.rmtree(destination)

# Copy dataset
shutil.copytree(path, destination)

print("Dataset berhasil disalin ke:", destination)
print("Isi dataset:", os.listdir(destination))

Path dataset asli: /Users/akbarrzql/.cache/kagglehub/datasets/albertnathaniel12/indonesian-spices-dataset/versions/11
Dataset berhasil disalin ke: /Users/akbarrzql/PycharmProjects/IndonesiaSpicesProject/data
Isi dataset: ['Indonesian Spices Dataset']


## Split Dataset: Train / Validation / Test

Dataset asli berisi 31 kelas rempah dengan struktur `data/Indonesian Spices Dataset/<nama kelas>/*.jpg`.

Split dilakukan secara **stratified** (proporsi tiap kelas sama di semua subset) dengan rasio **70% train / 15% validation / 15% test**.
Hasilnya ditulis ke `data_split/` dengan format `ImageFolder` sehingga langsung bisa dipakai `torchvision.datasets.ImageFolder`.

> Catatan: folder output sengaja dipisah dari `data/`, karena cell download di atas menghapus isi `data/` setiap kali dijalankan.

In [7]:
from pathlib import Path
import shutil
import random

import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

PROJECT_DIR = Path("/Users/akbarrzql/PycharmProjects/IndonesiaSpicesProject")
SOURCE_DIR = PROJECT_DIR / "data" / "Indonesian Spices Dataset"
OUTPUT_DIR = PROJECT_DIR / "data_split"

TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
SEED = 42
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9, "Total rasio harus 1.0"
assert SOURCE_DIR.exists(), f"Dataset tidak ditemukan di {SOURCE_DIR}. Jalankan cell download di atas."

random.seed(SEED)

# Kumpulkan semua file gambar beserta labelnya
records = []
for class_dir in sorted(p for p in SOURCE_DIR.iterdir() if p.is_dir()):
    for img_path in sorted(class_dir.rglob("*")):
        if img_path.is_file() and img_path.suffix.lower() in IMG_EXTENSIONS:
            records.append({"filepath": img_path, "label": class_dir.name})

df = pd.DataFrame(records)

print(f"Jumlah kelas : {df['label'].nunique()}")
print(f"Jumlah gambar: {len(df)}")
print("\nDistribusi per kelas:")
print(df["label"].value_counts().sort_index())

Jumlah kelas : 31
Jumlah gambar: 6510

Distribusi per kelas:
label
adas             210
andaliman        210
asam jawa        210
bawang bombai    210
bawang merah     210
bawang putih     210
biji ketumbar    210
bukan rempah     210
bunga lawang     210
cengkeh          210
daun jeruk       210
daun kemangi     210
daun ketumbar    210
daun salam       210
jahe             210
jinten           210
kapulaga         210
kayu manis       210
kayu secang      210
kemiri           210
kemukus          210
kencur           210
kluwek           210
kunyit           210
lada             210
lengkuas         210
pala             210
saffron          210
serai            210
vanili           210
wijen            210
Name: count, dtype: int64


### Stratified split 70 / 15 / 15

Dilakukan dua tahap: pertama memisahkan train vs sisa, lalu sisa dibagi menjadi validation dan test.

In [8]:
# Tahap 1: train vs (val + test)
train_df, temp_df = train_test_split(
    df,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df["label"],
    random_state=SEED,
    shuffle=True,
)

# Tahap 2: val vs test (proporsional terhadap sisa)
val_df, test_df = train_test_split(
    temp_df,
    test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    stratify=temp_df["label"],
    random_state=SEED,
    shuffle=True,
)

splits = {"train": train_df, "val": val_df, "test": test_df}

for name, part in splits.items():
    print(f"{name:5s}: {len(part):5d} gambar ({len(part) / len(df):.1%})")

# Pastikan tidak ada file yang bocor antar subset
assert len(train_df) + len(val_df) + len(test_df) == len(df)
assert not (set(train_df['filepath']) & set(val_df['filepath']) & set(test_df['filepath']))

train:  4557 gambar (70.0%)
val  :   976 gambar (15.0%)
test :   977 gambar (15.0%)


### Copy file ke struktur `ImageFolder`

```
data_split/
├── train/<nama kelas>/*.jpg
├── val/<nama kelas>/*.jpg
└── test/<nama kelas>/*.jpg
```

In [9]:
# Mulai dari kondisi bersih agar hasil split tidak bercampur dengan run sebelumnya
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

manifest_rows = []

for split_name, part in splits.items():
    for label in df["label"].unique():
        (OUTPUT_DIR / split_name / label).mkdir(parents=True, exist_ok=True)

    for row in tqdm(part.itertuples(index=False), total=len(part), desc=f"copy {split_name}"):
        dest = OUTPUT_DIR / split_name / row.label / row.filepath.name
        shutil.copy2(row.filepath, dest)
        manifest_rows.append(
            {
                "split": split_name,
                "label": row.label,
                "filename": row.filepath.name,
                "filepath": str(dest.relative_to(PROJECT_DIR)),
            }
        )

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUTPUT_DIR / "manifest.csv", index=False)

print(f"Selesai. Hasil split: {OUTPUT_DIR}")
print(f"Manifest  : {OUTPUT_DIR / 'manifest.csv'} ({len(manifest)} baris)")

copy test: 100%|██████████| 977/977 [00:00<00:00, 2137.27it/s]

Selesai. Hasil split: /Users/akbarrzql/PycharmProjects/IndonesiaSpicesProject/data_split
Manifest  : /Users/akbarrzql/PycharmProjects/IndonesiaSpicesProject/data_split/manifest.csv (6510 baris)


### Verifikasi hasil split

In [10]:
# Hitung ulang langsung dari hasil di disk, bukan dari dataframe
counts = {
    split_name: {
        class_dir.name: sum(1 for f in class_dir.iterdir() if f.is_file())
        for class_dir in sorted((OUTPUT_DIR / split_name).iterdir())
        if class_dir.is_dir()
    }
    for split_name in ("train", "val", "test")
}

summary = pd.DataFrame(counts).fillna(0).astype(int)
summary["total"] = summary.sum(axis=1)

print(summary.to_string())
print("\nTotal per split:")
print(summary[["train", "val", "test"]].sum())
print(f"\nTotal keseluruhan: {summary['total'].sum()} (dataset asli: {len(df)})")

               train  val  test  total
adas             147   32    31    210
andaliman        147   31    32    210
asam jawa        147   31    32    210
bawang bombai    147   31    32    210
bawang merah     147   32    31    210
bawang putih     147   32    31    210
biji ketumbar    147   31    32    210
bukan rempah     147   31    32    210
bunga lawang     147   32    31    210
cengkeh          147   32    31    210
daun jeruk       147   31    32    210
daun kemangi     147   32    31    210
daun ketumbar    147   32    31    210
daun salam       147   32    31    210
jahe             147   31    32    210
jinten           147   32    31    210
kapulaga         147   32    31    210
kayu manis       147   32    31    210
kayu secang      147   31    32    210
kemiri           147   31    32    210
kemukus          147   31    32    210
kencur           147   31    32    210
kluwek           147   31    32    210
kunyit           147   32    31    210
lada             147   32

In [11]:
# Contoh pemakaian dengan torchvision ImageFolder
from torchvision import datasets, transforms

preview_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

for split_name in ("train", "val", "test"):
    ds = datasets.ImageFolder(OUTPUT_DIR / split_name, transform=preview_tf)
    print(f"{split_name:5s}: {len(ds)} gambar, {len(ds.classes)} kelas")

print("\nMapping kelas -> index:")
print(ds.class_to_idx)

train: 4557 gambar, 31 kelas
val  : 976 gambar, 31 kelas
test : 977 gambar, 31 kelas

Mapping kelas -> index:
{'adas': 0, 'andaliman': 1, 'asam jawa': 2, 'bawang bombai': 3, 'bawang merah': 4, 'bawang putih': 5, 'biji ketumbar': 6, 'bukan rempah': 7, 'bunga lawang': 8, 'cengkeh': 9, 'daun jeruk': 10, 'daun kemangi': 11, 'daun ketumbar': 12, 'daun salam': 13, 'jahe': 14, 'jinten': 15, 'kapulaga': 16, 'kayu manis': 17, 'kayu secang': 18, 'kemiri': 19, 'kemukus': 20, 'kencur': 21, 'kluwek': 22, 'kunyit': 23, 'lada': 24, 'lengkuas': 25, 'pala': 26, 'saffron': 27, 'serai': 28, 'vanili': 29, 'wijen': 30}
